In [ ]:
# ============================================================================
# TEXTILE PATTERN GENERATION - LoRA Training Pipeline
# Training time: ~5-7 hours on Kaggle T4 GPU
# ============================================================================

import sys
import os
import shutil
from pathlib import Path
import json
import random
from datetime import datetime
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import gc
import torch

print("="*80)
print("🎨 TEXTILE PATTERN LoRA TRAINER")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*80)

# Disable bitsandbytes to avoid CUDA issues
print("\n🔧 Configuring environment...")
import os
import sys

os.environ['BITSANDBYTES_NOWELCOME'] = '1'
os.environ['DISABLE_BITSANDBYTES'] = '1'

# Remove bitsandbytes from sys.modules if already loaded
for key in list(sys.modules.keys()):
    if 'bitsandbytes' in key:
        del sys.modules[key]

print("✅ Disabled bitsandbytes integration")

# Install required packages
print("\n📦 Managing dependencies...")

import subprocess

# First, uninstall bitsandbytes completely
try:
    print("  Uninstalling bitsandbytes...", end=" ")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "bitsandbytes"], 
                        stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    print("✅")
except:
    print("(skipped)")

# Install only essential packages
critical_packages = [
    "peft>=0.17.0",  # Required by diffusers
]

for package in critical_packages:
    try:
        print(f"  Installing {package.split('>=')[0].split('==')[0]}...", end=" ")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package], 
                            stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        print("✅")
    except Exception as e:
        print(f"⚠️ ({str(e)[:30]}...)")

print("✅ Dependencies ready!")
print("ℹ️  Using regular Adam optimizer\n")

# Imports
from diffusers import (
    StableDiffusionPipeline, 
    StableDiffusionImg2ImgPipeline,
    DDPMScheduler,
    AutoencoderKL,
    UNet2DConditionModel
)
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from accelerate import Accelerator
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

print("✅ All imports successful!\n")

ModuleNotFoundError: No module named 'tqdm'

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Dataset paths
    INPUT_DATASET = "/kaggle/input/Dataset_Augmented"  # Fixed: correct Kaggle path
    OUTPUT_DIR = "/kaggle/working/lora_training"
    
    # Styles to train
    STYLES = ['bandhani', 'batik', 'ikat']
    
    # Dataset size reduction (samples per style)
    MAX_IMAGES_PER_STYLE = 3000  # Reduced from 18K to 3K per style
    
    # Training hyperparameters
    RESOLUTION = 512
    TRAIN_BATCH_SIZE = 2  # Reduced for T4
    GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = 8
    NUM_TRAIN_EPOCHS = 4  # 4 epochs × 3000 images = ~1500 steps per style
    LEARNING_RATE = 1e-4
    LR_SCHEDULER = "cosine"
    LR_WARMUP_STEPS = 100
    
    # LoRA config
    LORA_RANK = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.1
    LORA_TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]
    
    # Checkpointing
    SAVE_EVERY_N_STEPS = 250
    MAX_CHECKPOINTS = 5  # Keep only 5 latest
    
    # Validation & monitoring
    VALIDATION_EVERY_N_STEPS = 250
    NUM_VALIDATION_IMAGES = 4
    
    # Model
    BASE_MODEL = "runwayml/stable-diffusion-v1-5"
    
    # Optimization
    MIXED_PRECISION = "fp16"
    GRADIENT_CHECKPOINTING = True
    USE_8BIT_ADAM = True  # Will be disabled if bitsandbytes is not available
    
    # Prompts for validation
    VALIDATION_PROMPTS = [
        "vibrant red and gold pattern with small circular dots",
        "intricate floral motifs with blue and white colors",
        "geometric zigzag pattern with purple and orange",
        "traditional textile with dense repeating pattern"
    ]
    
    # Seed
    SEED = 42

# Create directories
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{Config.OUTPUT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{Config.OUTPUT_DIR}/validation_images", exist_ok=True)
os.makedirs(f"{Config.OUTPUT_DIR}/plots", exist_ok=True)

# Check if bitsandbytes is available
try:
    import bitsandbytes
except ImportError:
    print("⚠️  bitsandbytes not available - disabling 8-bit Adam")
    Config.USE_8BIT_ADAM = False

# Set seed
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
torch.cuda.manual_seed_all(Config.SEED)

print("✅ Configuration loaded\n")
print(f"Dataset reduction: {Config.MAX_IMAGES_PER_STYLE} images per style")
print(f"Total images: {Config.MAX_IMAGES_PER_STYLE * len(Config.STYLES)}")
print(f"Expected training time: ~5-7 hours on T4\n")
print(f"8-bit Adam enabled: {Config.USE_8BIT_ADAM}\n")

In [ ]:
# ============================================================================
# CHECK KAGGLE INPUT DIRECTORY STRUCTURE
# ============================================================================

import os
from pathlib import Path

print("="*80)
print("📁 KAGGLE INPUT DIRECTORY STRUCTURE")
print("="*80)

kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print(f"\n✅ Kaggle input directory exists: {kaggle_input}\n")
    
    # List all datasets
    datasets = list(kaggle_input.iterdir())
    print(f"Found {len(datasets)} dataset(s):\n")
    
    for dataset_dir in datasets:
        print(f"📦 {dataset_dir.name}/")
        
        # List contents of each dataset (first 2 levels)
        try:
            contents = list(dataset_dir.iterdir())[:10]
            for item in contents:
                if item.is_dir():
                    print(f"  📁 {item.name}/")
                    # Show subdirectories
                    try:
                        subitems = list(item.iterdir())[:5]
                        for subitem in subitems:
                            if subitem.is_dir():
                                print(f"    📁 {subitem.name}/")
                            else:
                                print(f"    📄 {subitem.name}")
                    except:
                        pass
                else:
                    print(f"  📄 {item.name}")
        except Exception as e:
            print(f"  ⚠️  Error: {e}")
        print()
else:
    print("❌ Kaggle input directory not found!")
    print("Are you running this on Kaggle? If not, update Config.INPUT_DATASET to your local path")

print("="*80)

In [ ]:
# ============================================================================
# DATASET PREPARATION WITH SIZE REDUCTION
# ============================================================================

def prepare_reduced_dataset(input_dir, output_dir, max_per_style=3000):
    """
    Prepare dataset with size reduction:
    1. Resize images 256→512
    2. Sample max N images per style
    3. Export captions
    """
    
    print("="*80)
    print("📂 DATASET PREPARATION")
    print("="*80)
    
    # First, check what's actually in the input directory
    print(f"📁 Checking dataset structure at: {input_dir}")
    if os.path.exists(input_dir):
        print(f"   Directory exists! Contents:")
        try:
            items = os.listdir(input_dir)
            for item in items[:20]:  # Show first 20 items
                item_path = os.path.join(input_dir, item)
                if os.path.isdir(item_path):
                    print(f"   📁 {item}/")
                else:
                    print(f"   📄 {item}")
            if len(items) > 20:
                print(f"   ... and {len(items) - 20} more items")
        except Exception as e:
            print(f"   ⚠️  Error listing directory: {e}")
    else:
        print(f"   ❌ Directory does not exist!")
        print(f"   Checking parent directories...")
        parent = Path(input_dir).parent
        if parent.exists():
            print(f"   Parent exists: {parent}")
            print(f"   Contents: {list(parent.iterdir())[:10]}")
    print()
    
    metadata_csv = f"{input_dir}/dataset_metadata.csv"
    
    # If metadata doesn't exist, create it from the dataset structure
    if not os.path.exists(metadata_csv):
        print(f"⚠️  Metadata not found, generating from dataset structure...")
        
        all_data = []
        for style in Config.STYLES:
            style_dir = f"{input_dir}/{style}"
            if not os.path.exists(style_dir):
                print(f"   ⚠️  Style directory not found: {style_dir}")
                continue
            
            # Find all image files
            image_files = []
            for ext in ['*.jpg', '*.jpeg', '*.png']:
                image_files.extend(Path(style_dir).rglob(ext))
            
            print(f"   Found {len(image_files)} images for {style}")
            
            for img_path in image_files:
                # Check for corresponding caption file
                caption_path = img_path.with_suffix('.txt')
                caption = f"{style} textile pattern"  # default caption
                
                if caption_path.exists():
                    with open(caption_path, 'r', encoding='utf-8') as f:
                        caption = f.read().strip()
                
                all_data.append({
                    'filename': img_path.name,
                    'style': style,
                    'caption': caption,
                    'full_path': str(img_path)
                })
        
        if not all_data:
            print("❌ No images found in dataset!")
            return None
        
        df = pd.DataFrame(all_data)
        print(f"✅ Generated metadata for {len(df)} images\n")
    else:
        # Load existing CSV and check structure
        df = pd.read_csv(metadata_csv)
        print(f"📄 Loaded CSV with {len(df)} rows")
        print(f"   Columns: {', '.join(df.columns.tolist())}\n")
        
        # If 'style' column missing, infer from file path
        if 'style' not in df.columns:
            print("⚠️  'style' column missing, inferring from file paths...")
            
            def infer_style(row):
                # Try to get style from image_path or filename
                path = row.get('image_path', row.get('filename', ''))
                for style in Config.STYLES:
                    if style in path.lower():
                        return style
                return None
            
            df['style'] = df.apply(infer_style, axis=1)
            df = df[df['style'].notna()]  # Remove rows where style couldn't be inferred
            print(f"   Inferred style for {len(df)} images\n")
        
        # Ensure we have full_path for image loading
        if 'full_path' not in df.columns and 'image_path' in df.columns:
            df['full_path'] = df['image_path'].apply(lambda x: f"{input_dir}/{x}")
        elif 'full_path' not in df.columns:
            # Build full path from style and filename
            df['full_path'] = df.apply(lambda row: None, axis=1)  # Will search later
    
    selected_data = []
    
    for style in Config.STYLES:
        style_df = df[df['style'] == style]
        
        print(f"📁 {style.upper()}")
        print(f"   Original: {len(style_df)} images")
        
        # Sample if too many
        if len(style_df) > max_per_style:
            style_df = style_df.sample(n=max_per_style, random_state=Config.SEED)
            print(f"   Sampled: {len(style_df)} images")
        
        # Create output directory
        style_output = f"{output_dir}/{style}"
        os.makedirs(style_output, exist_ok=True)
        
        # Process images
        processed = 0
        for idx, row in tqdm(style_df.iterrows(), total=len(style_df), desc=f"   Processing {style}"):
            try:
                # Get filename - try different column names
                img_filename = row.get('filename', row.get('original_filename', None))
                img_path_rel = row.get('image_path', None)
                
                # Construct full path
                if 'full_path' in row and pd.notna(row['full_path']) and os.path.exists(row['full_path']):
                    img_path = row['full_path']
                elif img_path_rel:
                    # Build path from image_path column
                    img_path = f"{input_dir}/{img_path_rel}"
                elif img_filename:
                    # Search for image in subfolders
                    img_path = None
                    for root, dirs, files in os.walk(f"{input_dir}/{style}"):
                        if img_filename in files:
                            img_path = os.path.join(root, img_filename)
                            break
                else:
                    continue
                
                if img_path is None or not os.path.exists(img_path):
                    continue
                
                # Load and resize
                img = Image.open(img_path).convert('RGB')
                img_resized = img.resize((Config.RESOLUTION, Config.RESOLUTION), Image.Resampling.LANCZOS)
                
                # Save image
                # Generate output filename from original filename or image path
                if img_filename:
                    output_filename = img_filename
                else:
                    output_filename = os.path.basename(img_path_rel if img_path_rel else img_path)
                
                output_img_path = f"{style_output}/{output_filename}"
                img_resized.save(output_img_path, quality=95)
                
                # Save caption
                caption = row.get('caption', f"{style} textile pattern")
                caption_path = output_img_path.replace('.jpg', '.txt').replace('.png', '.txt')
                with open(caption_path, 'w', encoding='utf-8') as f:
                    f.write(caption)
                
                selected_data.append(row)
                processed += 1
                
            except Exception as e:
                # Print first few errors for debugging
                if processed == 0:
                    print(f"   ⚠️  Error: {str(e)[:100]}")
                continue
        
        print(f"   ✅ Processed: {processed} images\n")
    
    # Save reduced metadata
    reduced_df = pd.DataFrame(selected_data)
    reduced_df.to_csv(f"{output_dir}/dataset_reduced.csv", index=False)
    
    print("="*80)
    print(f"✅ Dataset prepared: {len(reduced_df)} total images")
    print(f"   Saved to: {output_dir}")
    print("="*80)
    
    return reduced_df

# Prepare dataset
PROCESSED_DATASET = f"{Config.OUTPUT_DIR}/dataset_prepared"
os.makedirs(PROCESSED_DATASET, exist_ok=True)

reduced_df = prepare_reduced_dataset(
    Config.INPUT_DATASET,
    PROCESSED_DATASET,
    max_per_style=Config.MAX_IMAGES_PER_STYLE
)

# Check what was actually created
if reduced_df is not None:
    print(f"\n📊 Dataset Summary:")
    print(f"   Total rows in CSV: {len(reduced_df)}")
    for style in Config.STYLES:
        style_dir = f"{PROCESSED_DATASET}/{style}"
        if os.path.exists(style_dir):
            files = [f for f in os.listdir(style_dir) if f.endswith(('.jpg', '.png'))]
            print(f"   {style}: {len(files)} images in {style_dir}")
        else:
            print(f"   {style}: directory not found!")
else:
    print("⚠️  Dataset preparation returned None!")

In [ ]:
# ============================================================================
# CUSTOM DATASET
# ============================================================================

class TextileDataset(Dataset):
    def __init__(self, data_dir, style, tokenizer, resolution=512):
        self.data_dir = data_dir
        self.style = style
        self.tokenizer = tokenizer
        self.resolution = resolution
        
        # Get all images for this style
        style_dir = f"{data_dir}/{style}"
        self.image_paths = []
        
        for file in os.listdir(style_dir):
            if file.endswith(('.jpg', '.png')):
                img_path = f"{style_dir}/{file}"
                caption_path = img_path.replace('.jpg', '.txt').replace('.png', '.txt')
                
                if os.path.exists(caption_path):
                    self.image_paths.append((img_path, caption_path))
        
        print(f"   Loaded {len(self.image_paths)} samples for {style}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path, caption_path = self.image_paths[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        image = image.resize((self.resolution, self.resolution), Image.Resampling.LANCZOS)
        image = np.array(image).astype(np.float32) / 127.5 - 1.0  # Normalize to [-1, 1]
        image = torch.from_numpy(image).permute(2, 0, 1)  # CHW
        
        # Load caption
        with open(caption_path, 'r', encoding='utf-8') as f:
            caption = f.read().strip()
        
        # Tokenize
        tokens = self.tokenizer(
            caption,
            truncation=True,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt"
        ).input_ids[0]
        
        return {
            "pixel_values": image,
            "input_ids": tokens
        }

print("✅ Dataset class defined\n")

In [ ]:
# ============================================================================
# CHECKPOINT MANAGEMENT (Keep only 5 latest)
# ============================================================================

def save_checkpoint(accelerator, unet, text_encoder, style, step, checkpoint_dir):
    """Save checkpoint and manage max checkpoints"""
    
    checkpoint_name = f"{style}_step_{step}"
    checkpoint_path = f"{checkpoint_dir}/{checkpoint_name}"
    
    # Save LoRA weights
    unwrapped_unet = accelerator.unwrap_model(unet)
    unwrapped_unet.save_pretrained(checkpoint_path)
    
    print(f"   💾 Checkpoint saved: {checkpoint_name}")
    
    # Manage checkpoint count
    cleanup_old_checkpoints(checkpoint_dir, style, max_keep=Config.MAX_CHECKPOINTS)

def cleanup_old_checkpoints(checkpoint_dir, style, max_keep=5):
    """Keep only the N most recent checkpoints for a style"""
    
    # Find all checkpoints for this style
    checkpoints = []
    for folder in os.listdir(checkpoint_dir):
        if folder.startswith(f"{style}_step_"):
            step = int(folder.split("_")[-1])
            checkpoints.append((step, folder))
    
    # Sort by step (newest first)
    checkpoints.sort(reverse=True, key=lambda x: x[0])
    
    # Delete old checkpoints
    if len(checkpoints) > max_keep:
        for step, folder in checkpoints[max_keep:]:
            path = f"{checkpoint_dir}/{folder}"
            shutil.rmtree(path, ignore_errors=True)
            print(f"   🗑️  Deleted old checkpoint: {folder}")

def find_latest_checkpoint(checkpoint_dir, style):
    """Find the most recent checkpoint for resuming"""
    
    checkpoints = []
    for folder in os.listdir(checkpoint_dir):
        if folder.startswith(f"{style}_step_"):
            step = int(folder.split("_")[-1])
            checkpoints.append((step, folder))
    
    if not checkpoints:
        return None, 0
    
    checkpoints.sort(reverse=True, key=lambda x: x[0])
    latest_step, latest_folder = checkpoints[0]
    
    return f"{checkpoint_dir}/{latest_folder}", latest_step

print("✅ Checkpoint management ready\n")

In [ ]:
# ============================================================================
# VALIDATION & VISUALIZATION
# ============================================================================

def generate_validation_images(pipeline, style, step, prompts, save_dir):
    """Generate validation images during training"""
    
    pipeline.to("cuda")
    pipeline.set_progress_bar_config(disable=True)
    
    images = []
    
    for prompt in prompts:
        full_prompt = f"traditional {style} textile pattern, {prompt}, seamless tileable pattern, high quality, detailed, no text, no watermarks"
        negative_prompt = "blurry, low quality, distorted, human, face, people, text, watermark"
        
        with torch.no_grad():
            try:
                result = pipeline(
                    prompt=full_prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=25,
                    guidance_scale=7.5,
                    height=512,
                    width=512
                )
                image = result.images[0]
            except Exception as e:
                # If generation fails, create a placeholder
                print(f"   ⚠️  Error generating image: {str(e)[:50]}")
                image = Image.new('RGB', (512, 512), color='gray')
        
        images.append(image)
    
    # Create grid
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()
    
    for i, (img, prompt) in enumerate(zip(images, prompts)):
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(prompt[:40], fontsize=10)
    
    plt.suptitle(f"{style.upper()} - Step {step}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    save_path = f"{save_dir}/{style}_step_{step}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"   🖼️  Validation images saved: {save_path}")
    
    return images

def plot_training_metrics(losses, style, save_dir):
    """Plot loss curves"""
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(losses, linewidth=2, color='#2E86AB')
    plt.title(f'{style.upper()} - Training Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    # Moving average
    window = 50
    if len(losses) >= window:
        ma = np.convolve(losses, np.ones(window)/window, mode='valid')
        plt.plot(ma, linewidth=2, color='#A23B72', label=f'{window}-step MA')
        plt.title(f'{style.upper()} - Smoothed Loss', fontsize=14, fontweight='bold')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(alpha=0.3)
    
    plt.tight_layout()
    save_path = f"{save_dir}/{style}_loss_plot.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"   📈 Loss plot saved: {save_path}")

print("✅ Validation functions ready\n")

In [ ]:
# ============================================================================
# MAIN TRAINING FUNCTION
# ============================================================================

def train_lora_for_style(style, resume_from_checkpoint=True):
    """
    Train LoRA for one style with all features:
    - Checkpoint saving & resuming
    - Validation image generation
    - Loss plotting
    - Progress monitoring
    """
    
    print("="*80)
    print(f"🎨 TRAINING LoRA FOR: {style.upper()}")
    print("="*80)
    
    # Initialize accelerator
    accelerator = Accelerator(
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
        mixed_precision=Config.MIXED_PRECISION
    )
    
    # Load tokenizer
    tokenizer = CLIPTokenizer.from_pretrained(Config.BASE_MODEL, subfolder="tokenizer")
    
    # Load models
    print("\n📦 Loading base model...")
    text_encoder = CLIPTextModel.from_pretrained(Config.BASE_MODEL, subfolder="text_encoder")
    vae = AutoencoderKL.from_pretrained(Config.BASE_MODEL, subfolder="vae")
    unet = UNet2DConditionModel.from_pretrained(Config.BASE_MODEL, subfolder="unet")
    
    # Freeze VAE and text encoder
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    
    # Enable gradient checkpointing
    if Config.GRADIENT_CHECKPOINTING:
        unet.enable_gradient_checkpointing()
    
    # Configure LoRA
    print("🔧 Configuring LoRA...")
    lora_config = LoraConfig(
        r=Config.LORA_RANK,
        lora_alpha=Config.LORA_ALPHA,
        init_lora_weights="gaussian",
        target_modules=Config.LORA_TARGET_MODULES,
        lora_dropout=Config.LORA_DROPOUT,
    )
    
    unet = get_peft_model(unet, lora_config)
    unet.print_trainable_parameters()
    
    # Check for existing checkpoint
    resume_step = 0
    checkpoint_path, resume_step = find_latest_checkpoint(f"{Config.OUTPUT_DIR}/checkpoints", style)
    
    if resume_from_checkpoint and checkpoint_path:
        print(f"\n♻️  Resuming from checkpoint: {checkpoint_path} (step {resume_step})")
        unet = PeftModel.from_pretrained(unet, checkpoint_path)
    else:
        print("\n🆕 Starting fresh training")
    
    # Optimizer
    if Config.USE_8BIT_ADAM:
        try:
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(unet.parameters(), lr=Config.LEARNING_RATE)
            print("✅ Using 8-bit Adam optimizer")
        except ImportError:
            print("⚠️  bitsandbytes not available, falling back to regular AdamW")
            optimizer = torch.optim.AdamW(unet.parameters(), lr=Config.LEARNING_RATE)
    else:
        optimizer = torch.optim.AdamW(unet.parameters(), lr=Config.LEARNING_RATE)
        print("✅ Using regular AdamW optimizer")
    
    # Dataset
    print(f"\n📂 Loading dataset for {style}...")
    dataset = TextileDataset(PROCESSED_DATASET, style, tokenizer, Config.RESOLUTION)
    dataloader = DataLoader(
        dataset,
        batch_size=Config.TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )
    
    # Scheduler
    num_update_steps_per_epoch = len(dataloader) // Config.GRADIENT_ACCUMULATION_STEPS
    max_train_steps = Config.NUM_TRAIN_EPOCHS * num_update_steps_per_epoch
    
    from diffusers.optimization import get_scheduler
    lr_scheduler = get_scheduler(
        Config.LR_SCHEDULER,
        optimizer=optimizer,
        num_warmup_steps=Config.LR_WARMUP_STEPS,
        num_training_steps=max_train_steps
    )
    
    # Prepare with accelerator
    unet, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        unet, optimizer, dataloader, lr_scheduler
    )
    
    # Move models to device
    vae.to(accelerator.device)
    text_encoder.to(accelerator.device)
    
    # Noise scheduler
    noise_scheduler = DDPMScheduler.from_pretrained(Config.BASE_MODEL, subfolder="scheduler")
    
    # Training state
    global_step = resume_step
    losses = []
    
    print(f"\n🚀 Starting training...")
    print(f"   Total steps: {max_train_steps}")
    print(f"   Epochs: {Config.NUM_TRAIN_EPOCHS}")
    print(f"   Batch size: {Config.TRAIN_BATCH_SIZE}")
    print(f"   Gradient accumulation: {Config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"   Effective batch size: {Config.TRAIN_BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS}")
    print("="*80)
    
    # Training loop
    progress_bar = tqdm(range(global_step, max_train_steps), desc=f"Training {style}")
    
    for epoch in range(Config.NUM_TRAIN_EPOCHS):
        unet.train()
        
        for batch in dataloader:
            with accelerator.accumulate(unet):
                # Get latents
                latents = vae.encode(batch["pixel_values"].to(accelerator.device)).latent_dist.sample()
                latents = latents * vae.config.scaling_factor
                
                # Sample noise
                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                
                # Sample timesteps
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps, (bsz,),
                    device=latents.device
                ).long()
                
                # Add noise
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                
                # Get text embeddings
                encoder_hidden_states = text_encoder(batch["input_ids"].to(accelerator.device))[0]
                
                # Predict noise
                model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
                
                # Compute loss
                loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
                
                # Backprop
                accelerator.backward(loss)
                
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
            
            # Update progress
            if accelerator.sync_gradients:
                progress_bar.update(1)
                global_step += 1
                losses.append(loss.detach().item())
                
                progress_bar.set_postfix({
                    'loss': f'{loss.detach().item():.4f}',
                    'lr': f'{lr_scheduler.get_last_lr()[0]:.2e}'
                })
                
                # Save checkpoint
                if global_step % Config.SAVE_EVERY_N_STEPS == 0:
                    save_checkpoint(
                        accelerator, unet, text_encoder, style, global_step,
                        f"{Config.OUTPUT_DIR}/checkpoints"
                    )
                
                # Validation
                if global_step % Config.VALIDATION_EVERY_N_STEPS == 0:
                    print(f"\n   🖼️  Generating validation images at step {global_step}...")
                    
                    # Unwrap LoRA to get base UNet model
                    unwrapped_unet = accelerator.unwrap_model(unet)
                    
                    # Get base model from PeftModel if it's wrapped
                    if hasattr(unwrapped_unet, 'get_base_model'):
                        base_unet = unwrapped_unet.get_base_model()
                    else:
                        base_unet = unwrapped_unet
                    
                    # Create validation pipeline with safety checker disabled
                    pipeline = StableDiffusionPipeline.from_pretrained(
                        Config.BASE_MODEL,
                        unet=base_unet,
                        text_encoder=text_encoder,
                        vae=vae,
                        safety_checker=None,  # Disable NSFW checker for textile patterns
                        torch_dtype=torch.float16
                    )
                    
                    generate_validation_images(
                        pipeline, style, global_step,
                        Config.VALIDATION_PROMPTS,
                        f"{Config.OUTPUT_DIR}/validation_images"
                    )
                    
                    del pipeline
                    torch.cuda.empty_cache()
                
                if global_step >= max_train_steps:
                    break
        
        if global_step >= max_train_steps:
            break
    
    # Final save
    print(f"\n💾 Saving final LoRA...")
    final_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
    accelerator.unwrap_model(unet).save_pretrained(final_path)
    print(f"   ✅ Saved to: {final_path}")
    
    # Plot losses
    print(f"\n📊 Plotting training metrics...")
    plot_training_metrics(losses, style, f"{Config.OUTPUT_DIR}/plots")
    
    # Cleanup
    del unet, vae, text_encoder, optimizer, dataloader
    torch.cuda.empty_cache()
    gc.collect()
    
    print(f"\n{'='*80}")
    print(f"✅ TRAINING COMPLETE FOR {style.upper()}")
    print(f"{'='*80}\n")
    
    return losses

print("✅ Training function ready\n")

In [ ]:
# ============================================================================
# TRAIN ALL STYLES
# ============================================================================

training_results = {}

for style in Config.STYLES:
    try:
        losses = train_lora_for_style(style, resume_from_checkpoint=True)
        training_results[style] = {
            'status': 'success',
            'final_loss': losses[-1] if losses else None,
            'total_steps': len(losses)
        }
    except Exception as e:
        print(f"\n❌ Error training {style}: {e}")
        training_results[style] = {
            'status': 'failed',
            'error': str(e)
        }
        continue

# Summary
print("\n" + "="*80)
print("📊 TRAINING SUMMARY")
print("="*80)
for style, result in training_results.items():
    print(f"\n{style.upper()}:")
    print(f"   Status: {result['status']}")
    if result['status'] == 'success':
        print(f"   Final Loss: {result['final_loss']:.4f}")
        print(f"   Total Steps: {result['total_steps']}")
print("="*80)

In [ ]:
# ============================================================================
# EVALUATION & TESTING
# ============================================================================

print("="*80)
print("🧪 MODEL EVALUATION")
print("="*80)

def load_and_test_lora(style, test_prompts, test_image=None, strength=0.7):
    """
    Load trained LoRA and generate test images
    
    Args:
        style: Style name
        test_prompts: List of prompts
        test_image: Optional reference image (PIL Image or path)
        strength: img2img strength (if test_image provided)
    """
    
    print(f"\n🎨 Testing {style.upper()} LoRA")
    print("-" * 40)
    
    # Load base model
    if test_image is None:
        # Text-to-image
        pipeline = StableDiffusionPipeline.from_pretrained(
            Config.BASE_MODEL,
            torch_dtype=torch.float16,
            safety_checker=None
        )
    else:
        # Image-to-image
        pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
            Config.BASE_MODEL,
            torch_dtype=torch.float16,
            safety_checker=None
        )
    
    # Load LoRA weights
    lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
    
    if not os.path.exists(lora_path):
        print(f"❌ LoRA not found: {lora_path}")
        return []
    
    pipeline.load_lora_weights(lora_path)
    pipeline.to("cuda")
    
    results = []
    
    for i, prompt in enumerate(test_prompts):
        full_prompt = f"traditional {style} textile pattern, {prompt}, high quality, detailed fabric texture, seamless pattern"
        negative_prompt = "blurry, low quality, watermark, text, border, people, faces"
        
        print(f"\n   Prompt {i+1}: {prompt[:50]}...")
        
        if test_image is None:
            # Text-to-image
            image = pipeline(
                prompt=full_prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=50,
                guidance_scale=7.5,
                height=512,
                width=512
            ).images[0]
        else:
            # Image-to-image
            if isinstance(test_image, str):
                ref_img = Image.open(test_image).convert('RGB')
            else:
                ref_img = test_image
            
            ref_img = ref_img.resize((512, 512))
            
            image = pipeline(
                prompt=full_prompt,
                negative_prompt=negative_prompt,
                image=ref_img,
                strength=strength,
                num_inference_steps=50,
                guidance_scale=7.5
            ).images[0]
        
        results.append((prompt, image))
        
        # Save
        save_path = f"{Config.OUTPUT_DIR}/test_{style}_{i+1}.png"
        image.save(save_path)
        print(f"   ✅ Saved: {save_path}")
    
    del pipeline
    torch.cuda.empty_cache()
    
    return results

# Test prompts for each style
test_prompts_per_style = {
    'bandhani': [
        "vibrant red and gold with small circular dots in grid pattern",
        "blue and white tie-dye with dense dotted design",
        "multicolored circular patterns with geometric arrangement",
        "traditional orange and yellow bandhani with intricate dots"
    ],
    'batik': [
        "rich brown and indigo with detailed floral paisley motifs",
        "blue and gold batik with nature-inspired leaf patterns",
        "traditional Indonesian design with intricate wax-resist details",
        "multicolored batik with flowing organic shapes"
    ],
    'ikat': [
        "purple and orange geometric zigzag with blurred edges",
        "traditional tribal pattern with characteristic ikat technique",
        "bold diagonal stripes with handwoven texture",
        "red and blue geometric diamonds with soft edges"
    ]
}

# Test all styles (text-to-image)
all_results = {}

for style in Config.STYLES:
    prompts = test_prompts_per_style.get(style, Config.VALIDATION_PROMPTS)
    results = load_and_test_lora(style, prompts)
    all_results[style] = results

print("\n" + "="*80)
print("✅ TEXT-TO-IMAGE TESTING COMPLETE")
print("="*80)

# Create comparison grid
print("\n📊 Creating comparison grid...")

fig, axes = plt.subplots(len(Config.STYLES), 4, figsize=(20, len(Config.STYLES) * 5))

for row, style in enumerate(Config.STYLES):
    for col, (prompt, img) in enumerate(all_results[style][:4]):
        ax = axes[row, col] if len(Config.STYLES) > 1 else axes[col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(style.upper(), fontsize=14, fontweight='bold', rotation=0, labelpad=60, va='center')
        ax.set_title(prompt[:30] + "...", fontsize=9)

plt.suptitle('Model Evaluation - Text-to-Image Generation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{Config.OUTPUT_DIR}/evaluation_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Comparison grid saved: {Config.OUTPUT_DIR}/evaluation_comparison.png")

In [ ]:
# ============================================================================
# IMAGE-TO-IMAGE TESTING (Optional Reference Upload)
# ============================================================================

print("\n" + "="*80)
print("🖼️  IMAGE-TO-IMAGE TESTING")
print("="*80)

# Example: Use one of your dataset images as reference
# Or user can upload their own

def test_img2img_with_reference(style, reference_image_path, prompt, strength=0.7):
    """
    Test img2img with a reference image
    """
    
    print(f"\n🎨 Testing {style.upper()} with reference image")
    print(f"   Reference: {reference_image_path}")
    print(f"   Prompt: {prompt}")
    print(f"   Strength: {strength}")
    
    # Load pipeline
    pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
        Config.BASE_MODEL,
        torch_dtype=torch.float16,
        safety_checker=None
    )
    
    # Load LoRA
    lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
    pipeline.load_lora_weights(lora_path)
    pipeline.to("cuda")
    
    # Load reference
    ref_img = Image.open(reference_image_path).convert('RGB').resize((512, 512))
    
    # Generate
    full_prompt = f"traditional {style} textile pattern, {prompt}, high quality, detailed"
    
    result = pipeline(
        prompt=full_prompt,
        image=ref_img,
        strength=strength,
        num_inference_steps=50,
        guidance_scale=7.5
    ).images[0]
    
    # Display
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(ref_img)
    axes[0].set_title("Reference Image", fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(result)
    axes[1].set_title(f"Generated (strength={strength})", fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    plt.suptitle(f"{style.upper()} - Img2Img Test\n{prompt}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    save_path = f"{Config.OUTPUT_DIR}/img2img_test_{style}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"   ✅ Saved: {save_path}")
    
    del pipeline
    torch.cuda.empty_cache()
    
    return result

# Example: Pick a sample image from each style's dataset
for style in Config.STYLES:
    style_dir = f"{PROCESSED_DATASET}/{style}"
    sample_images = [f for f in os.listdir(style_dir) if f.endswith(('.jpg', '.png'))]
    
    if sample_images:
        reference_img = f"{style_dir}/{sample_images[0]}"
        
        test_img2img_with_reference(
            style,
            reference_img,
            prompt="vibrant red and gold colors with enhanced details",
            strength=0.7
        )

print("\n" + "="*80)
print("✅ IMAGE-TO-IMAGE TESTING COMPLETE")
print("="*80)

In [ ]:
# ============================================================================
# PACKAGE TRAINED MODELS FOR DOWNLOAD
# ============================================================================

print("\n" + "="*80)
print("📦 PACKAGING MODELS")
print("="*80)

# Create final package structure
final_package_dir = "/kaggle/working/textile_loras_trained"
os.makedirs(final_package_dir, exist_ok=True)

for style in Config.STYLES:
    lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
    
    if os.path.exists(lora_path):
        # Copy to package directory
        dest_path = f"{final_package_dir}/{style}_lora"
        shutil.copytree(lora_path, dest_path, dirs_exist_ok=True)
        print(f"✅ Packaged: {style}_lora")

# Copy evaluation images
shutil.copytree(
    f"{Config.OUTPUT_DIR}/validation_images",
    f"{final_package_dir}/validation_samples",
    dirs_exist_ok=True
)

# Copy plots
shutil.copytree(
    f"{Config.OUTPUT_DIR}/plots",
    f"{final_package_dir}/training_plots",
    dirs_exist_ok=True
)

# Save training summary
summary = {
    'training_date': datetime.now().isoformat(),
    'config': {
        'base_model': Config.BASE_MODEL,
        'resolution': Config.RESOLUTION,
        'lora_rank': Config.LORA_RANK,
        'lora_alpha': Config.LORA_ALPHA,
        'epochs': Config.NUM_TRAIN_EPOCHS,
        'batch_size': Config.TRAIN_BATCH_SIZE,
        'learning_rate': Config.LEARNING_RATE,
    },
    'results': training_results
}

with open(f"{final_package_dir}/training_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Package ready: {final_package_dir}")

# Zip for download
print("\n🗜️  Creating zip archive...")
shutil.make_archive(
    '/kaggle/working/textile_loras_trained',
    'zip',
    final_package_dir
)

zip_size = os.path.getsize('/kaggle/working/textile_loras_trained.zip') / (1024 * 1024)
print(f"✅ Archive created: textile_loras_trained.zip ({zip_size:.2f} MB)")

print("\n" + "="*80)
print("🎉 TRAINING PIPELINE COMPLETE!")
print("="*80)
print("\n📥 Download 'textile_loras_trained.zip' from the Output panel")
print("\nContents:")
print("  - 3 trained LoRAs (bandhani, batik, ikat)")
print("  - Validation samples")
print("  - Training plots")
print("  - Training summary (JSON)")
print("="*80)

In [ ]:
# ============================================================================
# ADVANCED ZIP OUTPUT MANAGEMENT
# ============================================================================

import zipfile
from pathlib import Path

print("="*80)
print("📦 ZIP OUTPUT MANAGEMENT")
print("="*80)

# ============================================================================
# Option 1: Zip Everything (All Outputs)
# ============================================================================

def zip_all_outputs():
    """Zip the entire output directory"""
    
    print("\n🗜️  Option 1: Zipping ALL outputs...")
    
    zip_path = "/kaggle/working/textile_lora_all_outputs.zip"
    
    shutil.make_archive(
        zip_path.replace('.zip', ''),
        'zip',
        Config.OUTPUT_DIR
    )
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"   ✅ Created: textile_lora_all_outputs.zip ({size_mb:.2f} MB)")
    
    return zip_path

# ============================================================================
# Option 2: Zip Only Trained Models (Lightweight)
# ============================================================================

def zip_models_only():
    """Zip only the final trained LoRA models (smallest size)"""
    
    print("\n🗜️  Option 2: Zipping MODELS ONLY (lightweight)...")
    
    zip_path = "/kaggle/working/textile_lora_models_only.zip"
    
    # Create temporary directory for models
    temp_dir = "/kaggle/working/temp_models"
    os.makedirs(temp_dir, exist_ok=True)
    
    for style in Config.STYLES:
        lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
        if os.path.exists(lora_path):
            shutil.copytree(lora_path, f"{temp_dir}/{style}_lora", dirs_exist_ok=True)
    
    # Zip
    shutil.make_archive(
        zip_path.replace('.zip', ''),
        'zip',
        temp_dir
    )
    
    # Cleanup
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"   ✅ Created: textile_lora_models_only.zip ({size_mb:.2f} MB)")
    print(f"   📁 Contains: {len(Config.STYLES)} LoRA models")
    
    return zip_path

# ============================================================================
# Option 3: Zip Models + Validation Images
# ============================================================================

def zip_models_and_validation():
    """Zip models and validation/test images for review"""
    
    print("\n🗜️  Option 3: Zipping MODELS + VALIDATION IMAGES...")
    
    zip_path = "/kaggle/working/textile_lora_models_validation.zip"
    
    temp_dir = "/kaggle/working/temp_models_val"
    os.makedirs(temp_dir, exist_ok=True)
    
    # Copy models
    for style in Config.STYLES:
        lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
        if os.path.exists(lora_path):
            shutil.copytree(lora_path, f"{temp_dir}/models/{style}_lora", dirs_exist_ok=True)
    
    # Copy validation images
    val_dir = f"{Config.OUTPUT_DIR}/validation_images"
    if os.path.exists(val_dir):
        shutil.copytree(val_dir, f"{temp_dir}/validation_images", dirs_exist_ok=True)
    
    # Copy test images
    for style in Config.STYLES:
        for i in range(1, 5):
            test_img = f"{Config.OUTPUT_DIR}/test_{style}_{i}.png"
            if os.path.exists(test_img):
                os.makedirs(f"{temp_dir}/test_images", exist_ok=True)
                shutil.copy(test_img, f"{temp_dir}/test_images/")
    
    # Zip
    shutil.make_archive(
        zip_path.replace('.zip', ''),
        'zip',
        temp_dir
    )
    
    # Cleanup
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"   ✅ Created: textile_lora_models_validation.zip ({size_mb:.2f} MB)")
    
    return zip_path

# ============================================================================
# Option 4: Zip Individual Style (Separate Archives)
# ============================================================================

def zip_individual_styles():
    """Create separate zip for each style"""
    
    print("\n🗜️  Option 4: Creating INDIVIDUAL ZIPS per style...")
    
    zip_paths = []
    
    for style in Config.STYLES:
        print(f"\n   Processing {style}...")
        
        zip_path = f"/kaggle/working/textile_lora_{style}.zip"
        temp_dir = f"/kaggle/working/temp_{style}"
        os.makedirs(temp_dir, exist_ok=True)
        
        # Copy model
        lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
        if os.path.exists(lora_path):
            shutil.copytree(lora_path, f"{temp_dir}/{style}_lora", dirs_exist_ok=True)
        
        # Copy validation images for this style
        val_dir = f"{Config.OUTPUT_DIR}/validation_images"
        if os.path.exists(val_dir):
            for img_file in os.listdir(val_dir):
                if style in img_file.lower():
                    shutil.copy(
                        f"{val_dir}/{img_file}",
                        f"{temp_dir}/validation/"
                    )
        
        # Copy test images
        for i in range(1, 5):
            test_img = f"{Config.OUTPUT_DIR}/test_{style}_{i}.png"
            if os.path.exists(test_img):
                os.makedirs(f"{temp_dir}/test_images", exist_ok=True)
                shutil.copy(test_img, f"{temp_dir}/test_images/")
        
        # Copy loss plot
        plot_file = f"{Config.OUTPUT_DIR}/plots/{style}_loss_plot.png"
        if os.path.exists(plot_file):
            os.makedirs(f"{temp_dir}/plots", exist_ok=True)
            shutil.copy(plot_file, f"{temp_dir}/plots/")
        
        # Zip
        shutil.make_archive(
            zip_path.replace('.zip', ''),
            'zip',
            temp_dir
        )
        
        # Cleanup
        shutil.rmtree(temp_dir, ignore_errors=True)
        
        size_mb = os.path.getsize(zip_path) / (1024 * 1024)
        print(f"   ✅ {style}: textile_lora_{style}.zip ({size_mb:.2f} MB)")
        
        zip_paths.append(zip_path)
    
    return zip_paths

# ============================================================================
# Option 5: Smart Compression with Exclusions
# ============================================================================

def zip_smart_compression(exclude_checkpoints=True, exclude_dataset=True):
    """
    Smart zip with selective exclusions to reduce size
    
    Args:
        exclude_checkpoints: Don't include intermediate checkpoints (keep only final)
        exclude_dataset: Don't include prepared dataset
    """
    
    print("\n🗜️  Option 5: SMART COMPRESSION (exclude unnecessary files)...")
    
    zip_path = "/kaggle/working/textile_lora_smart.zip"
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        
        for root, dirs, files in os.walk(Config.OUTPUT_DIR):
            
            # Skip checkpoints folder if requested
            if exclude_checkpoints and 'checkpoints' in root:
                continue
            
            # Skip dataset folder if requested
            if exclude_dataset and 'dataset_prepared' in root:
                continue
            
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, Config.OUTPUT_DIR)
                
                # Add to zip
                zipf.write(file_path, arcname)
                
        # Also add final models explicitly
        for style in Config.STYLES:
            lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
            if os.path.exists(lora_path):
                for root, dirs, files in os.walk(lora_path):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, Config.OUTPUT_DIR)
                        zipf.write(file_path, arcname)
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"   ✅ Created: textile_lora_smart.zip ({size_mb:.2f} MB)")
    print(f"   📦 Excluded: {'checkpoints, ' if exclude_checkpoints else ''}{'dataset' if exclude_dataset else ''}")
    
    return zip_path

# ============================================================================
# Option 6: Create Deployment Package (Production Ready)
# ============================================================================

def zip_deployment_package():
    """
    Create a production-ready deployment package with:
    - Models
    - Inference script
    - Config
    - Sample images
    - README
    """
    
    print("\n🗜️  Option 6: Creating DEPLOYMENT PACKAGE...")
    
    zip_path = "/kaggle/working/textile_lora_deployment.zip"
    temp_dir = "/kaggle/working/deployment_package"
    os.makedirs(temp_dir, exist_ok=True)
    
    # 1. Copy models
    models_dir = f"{temp_dir}/models"
    os.makedirs(models_dir, exist_ok=True)
    
    for style in Config.STYLES:
        lora_path = f"{Config.OUTPUT_DIR}/{style}_lora_final"
        if os.path.exists(lora_path):
            shutil.copytree(lora_path, f"{models_dir}/{style}_lora", dirs_exist_ok=True)
    
    # 2. Create inference script
    inference_script = f"""
# Textile Pattern Generator - Inference Script
# Auto-generated deployment package

import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from PIL import Image

STYLES = {Config.STYLES}
BASE_MODEL = "{Config.BASE_MODEL}"

def generate_pattern(
    style,
    prompt,
    color=None,
    reference_image=None,
    strength=0.7
):
    '''Generate textile pattern'''
    
    # Load appropriate pipeline
    if reference_image is None:
        pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    else:
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    
    # Load LoRA
    pipe.load_lora_weights(f"models/{{style}}_lora")
    pipe.to("cuda")
    
    # Build prompt
    full_prompt = f"traditional {{style}} textile pattern, {{prompt}}"
    if color:
        full_prompt += f", {{color}} colors"
    full_prompt += ", high quality, detailed"
    
    # Generate
    if reference_image is None:
        image = pipe(prompt=full_prompt, num_inference_steps=50, guidance_scale=7.5).images[0]
    else:
        ref_img = Image.open(reference_image).convert('RGB').resize((512, 512))
        image = pipe(prompt=full_prompt, image=ref_img, strength=strength, num_inference_steps=50).images[0]
    
    return image

# Example usage:
# image = generate_pattern("bandhani", "vibrant red and gold with dots", color="red and gold")
# image.save("output.png")
"""
    
    with open(f"{temp_dir}/inference.py", 'w') as f:
        f.write(inference_script)
    
    # 3. Create README
    readme = f"""
# Textile Pattern Generator - Deployment Package

## Contents
- `models/` - Trained LoRA models ({len(Config.STYLES)} styles)
- `inference.py` - Ready-to-use inference script
- `config.json` - Training configuration
- `samples/` - Sample generated images

## Styles
{', '.join(Config.STYLES)}

## Quick Start

```python
from inference import generate_pattern

# Generate from text
image = generate_pattern(
    style="bandhani",
    prompt="vibrant red and gold with small dots",
    color="red and gold"
)
image.save("output.png")

# Generate from reference
image = generate_pattern(
    style="batik",
    prompt="enhance colors and details",
    reference_image="reference.jpg",
    strength=0.7
)